### Por que um novo notebook?
```text
        01_entendimento_dados
                ↓
        "O que existe na minha base?"

        02_preparacao_dados
                ↓
        "Como vou preparar esses dados para o modelo?"
```

In [1]:
# 1. Importações

from sklearn.model_selection import train_test_split

from consumer_credit_risk.data import carregar_dados_credito

import pandas as pd

In [2]:
# 2. Carregar os dados

df = carregar_dados_credito()

df.shape

(150000, 11)

In [3]:
# 3. Separar variáveis explicativas (X) e variável-alvo (y)

X = df.drop(columns="inadimplente_2anos")
y = df["inadimplente_2anos"]

Conceitualmente estamos separando as **variáveis explicativas** da **variável-alvo**.

```text
              BASE ORIGINAL
             150.000 × 11
                   │
          ┌────────┴────────┐
          ↓                 ↓
          X                 y
  características       alvo
     10 colunas         1 coluna
          │                 │
 idade                    0/1
 renda_mensal
 dependentes
 uso_limite...
 ...
 ```

In [4]:
# 4. Conferir as dimensões de X e y

print("X:", X.shape)
print("y:", y.shape)

X: (150000, 10)
y: (150000,)


Explicando o que faremos nos próximos passos
```text
    1. Separar treino/teste                  
            ↓
    2. Criar cópias                         
            ↓
    3. Identificar quem tem renda ausente
            ↓
    4. Criar renda_estimada = 0/1
            ↓
    5. Calcular mediana da renda NO TREINO
            ↓
    6. Preencher renda no treino e teste
            ↓
    7. Identificar dependentes ausentes
            ↓
    8. Criar dependentes_estimados = 0/1
            ↓
    9. Calcular mediana dos dependentes NO TREINO
            ↓
    10. Preencher dependentes
```

In [5]:
# 5. Separar os dados em treino e teste

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [6]:
# 6. Conferir o tamanho das bases

print("X_treino:", X_treino.shape)
print("X_teste:", X_teste.shape)
print("y_treino:", y_treino.shape)
print("y_teste:", y_teste.shape)

X_treino: (112500, 10)
X_teste: (37500, 10)
y_treino: (112500,)
y_teste: (37500,)


In [7]:
# 7. Conferir a proporção de inadimplentes

print("Base original:")
print(y.value_counts(normalize=True) * 100)

print("\nTreino:")
print(y_treino.value_counts(normalize=True) * 100)

print("\nTeste:")
print(y_teste.value_counts(normalize=True) * 100)

Base original:
inadimplente_2anos
0    93.316
1     6.684
Name: proportion, dtype: float64

Treino:
inadimplente_2anos
0    93.315556
1     6.684444
Name: proportion, dtype: float64

Teste:
inadimplente_2anos
0    93.317333
1     6.682667
Name: proportion, dtype: float64


O próximo passo será muito importante: começaremos o tratamento dos ausentes, mas calculando tudo **somente a partir** de `X_treino`. É aí que entra de verdade o conceito de evitar **data leakage**.

Agora entramos numa parte central da preparação: **tratar os valores ausentes sem deixar o conjunto de teste influenciar nossas decisões**.

Já temos:

```text
    150.000 registros
            │
            ├── Treino: 112.500 (75%)
            └── Teste:   37.500 (25%)
```
A partir de agora, o **treino ensina** e o **teste apenas recebe as regras aprendidas**.

In [8]:
# 8. Criar cópias das bases antes dos tratamentos

X_treino = X_treino.copy()
X_teste = X_teste.copy()

In [9]:
# 9. Identificar as rendas que precisarão ser estimadas

X_treino["renda_estimada"] = X_treino["renda_mensal"].isna().astype(int)
X_teste["renda_estimada"] = X_teste["renda_mensal"].isna().astype(int)

In [10]:
# 10. Conferir o indicador de renda estimada

X_treino["renda_estimada"].value_counts()

renda_estimada
0    90313
1    22187
Name: count, dtype: int64

In [11]:
# 11. Calcular a mediana da renda usando somente a base de treino

mediana_renda = X_treino["renda_mensal"].median()

mediana_renda

np.float64(5400.0)

Aqui existe um detalhe metodológico **muito importante**.

Temos três possibilidades aparentemente razoáveis:
```python
df["renda_mensal"].median()        # NÃO
X_teste["renda_mensal"].median()   # NÃO
X_treino["renda_mensal"].median()  # SIM
```

Por quê?

Porque estamos simulando o que aconteceria no mundo real:
```text
    TREINO
    dados que conhecemos
        │
        │ aprende a regra
        ▼
    mediana da renda
        │
        ├─────────────┐
        ▼             ▼
    TREINO         TESTE

```

O conjunto de teste deve permanecer como se fossem **clientes futuros que ainda não vimos**. Portanto, ele não pode nos ajudar a decidir qual valor usar para preencher a renda.

Um detalhe importante sobre `median()`

Você pode estar pensando:

> "Mas `renda_mensal` ainda tem vários `NaN`. Como o Python calcula a mediana?"

Por padrão, o pandas **ignora os valores ausentes** ao calcular a mediana.

Então:
```text
    3000
    4000
    NaN
    5000
    NaN
    6000
```
para calcular a mediana, ele considera os valores disponíveis:
```text
    3000
    4000
    5000
    6000
```
Os `NaN` não entram no cálculo.

In [12]:
# 12. Preencher rendas ausentes com a mediana calculada no treino

X_treino["renda_mensal"] = X_treino["renda_mensal"].fillna(mediana_renda)
X_teste["renda_mensal"] = X_teste["renda_mensal"].fillna(mediana_renda)

Repare em algo fundamental:
```text
    mediana calculada
        ↓
    X_treino
        ↓
        5400
        ↙  ↘
    treino  teste
```
Nós **não calculamos uma mediana diferente para o teste**. O teste recebe a regra aprendida no treino.

E agora nossas duas colunas trabalham juntas. Um cliente que originalmente tinha renda ausente passa, por exemplo, de:
```text
    renda_mensal    renda_estimada
    NaN             1
```
para:
```text
    renda_mensal    renda_estimada
    5400            1
```
Enquanto alguém cuja renda verdadeira já era R$ 5.400 fica:
```text
    renda_mensal    renda_estimada
    5400            0
```
É exatamente por isso que criamos **renda_estimada** antes do `fillna()`.

In [13]:
# 13. Conferir se ainda existem rendas ausentes

print("Treino:", X_treino["renda_mensal"].isna().sum())
print("Teste:", X_teste["renda_mensal"].isna().sum())

Treino: 0
Teste: 0


In [14]:
# 14. Identificar os dependentes que precisarão ser estimados

X_treino["dependentes_estimados"] = X_treino["dependentes"].isna().astype(int)
X_teste["dependentes_estimados"] = X_teste["dependentes"].isna().astype(int)

In [15]:
# 15. Conferir o indicador de dependentes estimados

X_treino["dependentes_estimados"].value_counts()

dependentes_estimados
0    109555
1      2945
Name: count, dtype: int64

Mesma lógia para que foi utilizada para renda.

```text
      X_treino["dependentes"]
            ↓
      calcular mediana
            ↓
      mediana_dependentes
            ↓
         ┌──┴──┐
         ↓     ↓
      treino  teste
```

In [16]:
# 16. Calcular a mediana dos dependentes usando somente a base de treino

mediana_dependentes = X_treino["dependentes"].median()

mediana_dependentes

np.float64(0.0)

In [17]:
# 17. Preencher dependentes ausentes com a mediana calculada no treino

X_treino["dependentes"] = X_treino["dependentes"].fillna(mediana_dependentes)
X_teste["dependentes"] = X_teste["dependentes"].fillna(mediana_dependentes)

In [18]:
# 18. Conferir se ainda existem dependentes ausentes

print("Treino:", X_treino["dependentes"].isna().sum())
print("Teste:", X_teste["dependentes"].isna().sum())

Treino: 0
Teste: 0


In [19]:
# 19. Analisar a distribuição das colunas de atraso na base de treino

colunas_atraso = [
    "atrasos_30_59_dias",
    "atrasos_60_89_dias",
    "atrasos_90_mais_dias"
]

X_treino[colunas_atraso].describe().T

,count,mean,std,min,25%,50%,75%,max
atrasos_30_59_dias,112500.0,0.421351,4.196450,0.0,0.0,0.0,0.0,98.0
atrasos_60_89_dias,112500.0,0.240676,4.158755,0.0,0.0,0.0,0.0,98.0
atrasos_90_mais_dias,112500.0,0.266764,4.173038,0.0,0.0,0.0,0.0,98.0


Esse resultado mostra por que o `describe()` sozinho não é suficiente para decidir o corte.

Observe as três colunas:
```text
Variável	Média	Mediana	75%	Máximo
30–59 dias	0,421	    0	 0	 98
60–89 dias	0,241	    0	 0	 98
90+ dias	0,267	    0	 0	 98
```
O fato de **75% ser zero**, mas o desvio-padrão estar perto de `4,2` e o máximo ser `98`, indica uma distribuição extremamente assimétrica. Os valores muito altos estão puxando a média e o desvio-padrão.

Agora precisamos enxergar **o que existe entre 0 e 98**.

In [20]:
# 20. Analisar os valores das ocorrências de atraso de 30 a 59 dias

X_treino["atrasos_30_59_dias"].value_counts().sort_index()

atrasos_30_59_dias
0     94474
1     12077
2      3447
3      1330
4       535
5       252
6       104
7        41
8        22
9         8
10        4
11        1
12        2
13        1
96        1
98      201
Name: count, dtype: int64

Isso reforça bastante que `96` e `98` têm comportamento diferente dos valores normais. Mas antes de decidir o tratamento, quero testar aquela informação da instrução original: **se quem tem 96/98 em uma coluna também tem esse código nas outras três**.

In [21]:
# 21. Verificar os registros com códigos 96 ou 98 nas colunas de atraso

mascara_96_98 = X_treino[colunas_atraso].isin([96, 98]).any(axis=1)

X_treino.loc[mascara_96_98, colunas_atraso].value_counts()

atrasos_30_59_dias  atrasos_60_89_dias  atrasos_90_mais_dias
98                  98                  98                      201
96                  96                  96                        1
Name: count, dtype: int64

### **E sobre o corte em 20?**

Temos uma descoberta interessante: nessa primeira coluna, **não existe nenhum valor normal entre 14 e 95**.

Portanto, um corte em `20` faria:
```text
    0–13 → permanecem exatamente iguais
    96   → 20
    98   → 20
```
Ou seja, pelo menos para essa coluna, o `20` não está realmente "cortando outliers normais". Ele está essencialmente substituindo os códigos especiais 96/98 por um teto.

Isso nos leva a uma questão metodológica que quero analisar antes de manter o `20`: talvez existam alternativas conceitualmente melhores para tratar `96/98`, já que eles parecem ser códigos especiais, e não simplesmente valores extremos.

In [22]:
# 22. Analisar os maiores valores de atraso, desconsiderando os códigos 96 e 98

for coluna in colunas_atraso:
    valores_normais = X_treino.loc[
        ~X_treino[coluna].isin([96, 98]),
        coluna
    ]

    print(f"{coluna}:")
    print(valores_normais.value_counts().sort_index().tail(10))
    print()

atrasos_30_59_dias:
atrasos_30_59_dias
4     535
5     252
6     104
7      41
8      22
9       8
10      4
11      1
12      2
13      1
Name: count, dtype: int64

atrasos_60_89_dias:
atrasos_60_89_dias
0     106775
1       4330
2        828
3        241
4         78
5         27
6         11
7          7
11         1
Name: count, dtype: int64

atrasos_90_mais_dias:
atrasos_90_mais_dias
7     27
8     15
9     14
10     6
11     3
12     1
13     3
14     2
15     2
17     1
Name: count, dtype: int64



In [23]:
# 23. Verificar o maior valor normal em cada coluna de atraso

for coluna in colunas_atraso:
    maximo_normal = X_treino.loc[
        ~X_treino[coluna].isin([96, 98]),
        coluna
    ].max()

    print(f"{coluna}: {maximo_normal}")

atrasos_30_59_dias: 13
atrasos_60_89_dias: 11
atrasos_90_mais_dias: 17


### **Então 20 é um bom corte?**

Para esta base, **é uma escolha coerente**, mas eu faria uma distinção importante: não estamos escolhendo 20 porque ele seja um quartil ou percentil ótimo. Estamos escolhendo um valor **acima do maior atraso normal observado (17)** e muito abaixo dos códigos anômalos `96/98`.

Poderíamos escolher `18`, `20`, `25` etc. Todos separariam os códigos especiais dos valores normais. O `20` tem a vantagem de ser um teto simples e interpretável.

Eu manteria o critério que já havíamos definido:
```text
    atrasos normais: 0–17 → preservados
    códigos 96/98        → 20
    indicador especial   → atrasos_96_98 = 1
```
Mas há uma melhoria importante para o código definitivo: em vez de simplesmente fazer `clip(upper=20)`, podemos deixar explícito que estamos tratando **96/98**, porque é isso que nossa análise justificou. Assim, se no futuro aparecer um atraso legítimo de 21, não o transformaríamos automaticamente em 20.

In [24]:
# 24. Criar indicador para clientes com códigos especiais 96 ou 98

X_treino["atrasos_96_98"] = (
    X_treino[colunas_atraso]
    .isin([96, 98])
    .any(axis=1)
    .astype(int)
)

X_teste["atrasos_96_98"] = (
    X_teste[colunas_atraso]
    .isin([96, 98])
    .any(axis=1)
    .astype(int)
)

A lógica é a mesma que estudamos:

```text
    .isin([96, 98])
        ↓
    procura 96 ou 98

    .any(axis=1)
        ↓
    existe em alguma das 3 colunas daquela linha?

    .astype(int)
        ↓
    False → 0
    True  → 1
```
No treino esperamos exatamente:
```text
    0 → 112.298
    1 →     202
```
Podemos conferir.

In [25]:
# 25. Conferir o indicador de códigos especiais

X_treino["atrasos_96_98"].value_counts()

atrasos_96_98
0    112298
1       202
Name: count, dtype: int64

In [26]:
# 26. Substituir os códigos especiais 96 e 98 pelo valor 20

X_treino[colunas_atraso] = X_treino[colunas_atraso].replace(
    {96: 20, 98: 20}
)

X_teste[colunas_atraso] = X_teste[colunas_atraso].replace(
    {96: 20, 98: 20}
)

In [27]:
# 27. Conferir os maiores valores após o tratamento

X_treino[colunas_atraso].max()

atrasos_30_59_dias      20
atrasos_60_89_dias      20
atrasos_90_mais_dias    20
dtype: int64

In [28]:
# 28. Analisar a distribuição da renda na base de treino

X_treino["renda_mensal"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999]
)

count    1.125000e+05
mean     6.412448e+03
std      1.333266e+04
min      0.000000e+00
25%      3.900000e+03
50%      5.400000e+03
75%      7.400000e+03
90%      1.075000e+04
95%      1.350000e+04
99%      2.300000e+04
99.5%    3.083300e+04
99.9%    7.283766e+04
max      3.008750e+06
Name: renda_mensal, dtype: float64

Esses percentis respondem a perguntas muito úteis. Por exemplo, se o percentil 99 for R$ 25.000, significa que **99% dos clientes têm renda de até R$ 25.000** e apenas 1% está acima disso.

Já o `99.9%` mostra onde estão praticamente todos os clientes, deixando somente 0,1% acima.

Agora ficou muito interessante. A distribuição mostra claramente que temos uma cauda direita muito longa.

Traduzindo a notação científica:


| Medida        |            Renda |
| ------------- | ---------------: |
| Q1 — 25%      |         R$ 3.900 |
| Mediana — 50% |         R$ 5.400 |
| Q3 — 75%      |         R$ 7.400 |
| 90%           |        R$ 10.750 |
| 95%           |        R$ 13.500 |
| 99%           |        R$ 23.000 |
| 99,5%         |        R$ 30.833 |
| 99,9%         |     R$ 72.837,66 |
| Máximo        | **R$ 3.008.750** |

Isso mostra algo importante: o máximo de **R$ 3 milhões** está absurdamente distante da maior parte da distribuição. Ao mesmo tempo, cortar simplesmente em R$ 23 mil porque esse é o percentil 99 seria agressivo: estaríamos alterando 1% da base e provavelmente muitas rendas perfeitamente plausíveis.

O teto proposto inicialmente (pelo Lira) de R$ 50.000 está em algum lugar entre os percentis 99,5 e 99,9. Portanto, à primeira vista, ele parece bem menos arbitrário do que poderia parecer.

Agora vamos comparar com o critério do IQR.

In [29]:
# 29. Calcular o limite superior de renda pelo critério do IQR

q1_renda = X_treino["renda_mensal"].quantile(0.25)
q3_renda = X_treino["renda_mensal"].quantile(0.75)

iqr_renda = q3_renda - q1_renda
limite_iqr_renda = q3_renda + 1.5 * iqr_renda

print("Q1:", q1_renda)
print("Q3:", q3_renda)
print("IQR:", iqr_renda)
print("Limite superior:", limite_iqr_renda)

Q1: 3900.0
Q3: 7400.0
IQR: 3500.0
Limite superior: 12650.0


Pelos números que acabamos de obter, já podemos prever:
```text
    Q1 = 3.900
    Q3 = 7.400

    IQR = 7.400 - 3.900
    IQR = 3.500

    Limite = 7.400 + 1,5 × 3.500
    Limite = R$ 12.650
```
E aqui aparece um excelente exemplo de por que **não devemos aplicar mecanicamente a regra de outliers**.

R$ 12.650 seria considerado o limite superior pela regra clássica do IQR. Mas nossa própria distribuição mostra que o percentil 95 está em R$ 13.500. Portanto, um corte em R$ 12.650 alteraria ****mais de 5% dos clientes**.

Para renda, isso provavelmente seria excessivamente agressivo: uma renda de R$ 15 mil, R$ 20 mil ou R$ 30 mil não é necessariamente erro só porque estatisticamente está distante do centro da distribuição.

Então o IQR pode ser ótimo para **detectar valores incomuns**, mas isso não significa automaticamente que seja um bom **teto para winsorização**.

In [30]:
# 30. Comparar diferentes possibilidades de corte da renda

cortes_renda = [
    12650,   # Limite pelo IQR
    23000,   # Aproximadamente percentil 99
    30833,   # Aproximadamente percentil 99,5
    50000,   # Corte originalmente proposto
    72838    # Aproximadamente percentil 99,9
]

for corte in cortes_renda:
    quantidade = (X_treino["renda_mensal"] > corte).sum()
    percentual = quantidade / len(X_treino) * 100

    print(
        f"Corte R$ {corte:,.0f}: "
        f"{quantidade} clientes ({percentual:.3f}%)"
    )

Corte R$ 12,650: 6840 clientes (6.080%)
Corte R$ 23,000: 1118 clientes (0.994%)
Corte R$ 30,833: 562 clientes (0.500%)
Corte R$ 50,000: 225 clientes (0.200%)
Corte R$ 72,838: 113 clientes (0.100%)


Agora temos uma comparação bem concreta:

| Critério          | Clientes afetados | % do treino |
| ----------------- | ----------------: | ----------: |
| IQR — R$ 12.650   |             6.840 |      6,080% |
| P99 — R$ 23.000   |             1.118 |      0,994% |
| P99,5 — R$ 30.833 |               562 |      0,500% |
| **R$ 50.000**     |           **225** |  **0,200%** |
| P99,9 — R$ 72.838 |               113 |      0,100% |


 
Isso já mostra que **R$ 50 mil é um corte bastante conservador**: preserva 99,8% das rendas exatamente como estão e atua somente na cauda extrema.

Mas ainda quero responder a uma pergunta antes de escolher: **como são esses 225 valores acima de R$ 50 mil?**

In [31]:
# 31. Analisar as rendas acima de R$ 50.000

rendas_acima_50k = X_treino.loc[
    X_treino["renda_mensal"] > 50000,
    "renda_mensal"
]

rendas_acima_50k.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    2.250000e+02
mean     1.281178e+05
std      2.564914e+05
min      5.044000e+04
25%      5.990000e+04
50%      7.291600e+04
75%      9.650000e+04
90%      1.801418e+05
95%      2.804512e+05
99%      1.015510e+06
max      3.008750e+06
Name: renda_mensal, dtype: float64

Agora temos informação suficiente para avaliar melhor o corte de renda.

Entre os **225 clientes acima de R$ 50 mil**, a distribuição é aproximadamente:


| Medida  |            Renda |
| ------- | ---------------: |
| Menor   |        R$ 50.440 |
| 25%     |        R$ 59.900 |
| Mediana |        R$ 72.916 |
| 75%     |        R$ 96.500 |
| 90%     |       R$ 180.142 |
| 95%     |       R$ 280.451 |
| 99%     |     R$ 1.015.510 |
| Máximo  | **R$ 3.008.750** |


 

Isso mostra uma coisa importante: **não temos simplesmente alguns valores ligeiramente acima de R$ 50 mil.** A cauda continua se estendendo enormemente.

Por exemplo, dentro desses 225 casos, metade está abaixo de aproximadamente R$ 73 mil, mas o 1% superior já passa de **R$ 1 milhão**. O máximo chega a **R$ 3 milhões**.

**O que eu concluiria sobre os critérios**

O IQR de R$ 12.650 me parece inadequado como teto. Ele classificaria mais de 6% da base como valores a serem alterados e atingiria rendas perfeitamente plausíveis.

O P99 de R$ 23 mil também me parece agressivo para uma variável como renda. Ele alteraria cerca de 1.118 pessoas.

O P99,5, perto de R$ 30,8 mil, já é menos agressivo, mas ainda modifica 562 registros.

O P99,9, perto de R$ 72,8 mil, tem uma justificativa estatística elegante, mas permitiria que valores muito altos continuassem tendo bastante influência até esse ponto.

Já o teto de R$ 50 mil modifica somente:
```text
    225 / 112.500 = 0,20%
```
e ainda está bem acima de:
```text
    P99   = R$ 23.000
    P99,5 = R$ 30.833
```
Então, **R$ 50 mil parece um teto defensável para este projeto**, principalmente se o objetivo é reduzir a influência extrema sem comprimir uma parcela relevante da população.

Mas eu faria mais uma análise antes de bater o martelo.

**Falta uma pergunta importante: isso ajuda o modelo?**

Até agora estamos analisando somente a distribuição de `renda_mensal`. Só que nosso objetivo final não é deixar a distribuição "bonita"; é prever inadimplência.

Então eu gostaria de descobrir se os grupos de renda extremamente alta apresentam algum comportamento particular em relação a `y_treino`.

Podemos comparar, por exemplo:
```text
    até R$ 30.833
    R$ 30.833–50.000
    R$ 50.000–72.838
    acima de R$ 72.838
```
com a proporção de inadimplentes em cada faixa.

Isso é especialmente interessante porque temos y_treino separado. Assim continuamos respeitando nossa regra: **toda a decisão é feita somente com o conjunto de treino**.

In [32]:
# 32. Analisar inadimplência por faixas de renda na base de treino

analise_renda = X_treino[["renda_mensal"]].copy()

analise_renda["inadimplente"] = y_treino

In [33]:
# 33. Criar faixas de renda para análise

faixas = [
    0,
    30833,
    50000,
    72838,
    float("inf")
]

rotulos = [
    "Até 30.833",
    "30.833 a 50.000",
    "50.000 a 72.838",
    "Acima de 72.838"
]

analise_renda["faixa_renda"] = pd.cut(
    analise_renda["renda_mensal"],
    bins=faixas,
    labels=rotulos,
    include_lowest=True
)

Aqui usamos alguns pontos que acabamos de encontrar:
```text
    30.833 ≈ P99,5
    50.000 = corte que estamos avaliando
    72.838 ≈ P99,9
```
`float("inf")` significa infinito. Então a última faixa pega qualquer valor acima de R$ 72.838, inclusive aquele caso de R$ 3 milhões.

In [34]:
# 34. Comparar quantidade de clientes e inadimplência por faixa de renda

analise_renda.groupby(
    "faixa_renda",
    observed=True
).agg(
    clientes=("inadimplente", "size"),
    inadimplentes=("inadimplente", "sum"),
    taxa_inadimplencia=("inadimplente", "mean")
)

,clientes,inadimplentes,taxa_inadimplencia
faixa_renda,,,
Até 30.833,111938,7489,0.066903
30.833 a 50.000,337,17,0.050445
50.000 a 72.838,112,7,0.062500
Acima de 72.838,113,7,0.061947


A principal conclusão dessa análise: **não aparece uma mudança marcante na taxa de inadimplência entre essas faixas de renda**.

Convertendo para porcentagens:

| Faixa de renda     | Clientes | Inadimplentes |      Taxa |
| ------------------ | -------: | ------------: | --------: |
| Até R$ 30.833      |  111.938 |         7.489 | **6,69%** |
| R$ 30.833–50.000   |      337 |            17 | **5,04%** |
| R$ 50.000–72.838   |      112 |             7 | **6,25%** |
| Acima de R$ 72.838 |      113 |             7 | **6,19%** |


E a inadimplência geral da nossa base estava perto de **6,68%**. Então realmente os grupos estão relativamente próximos.

Há uma ressalva importante: as faixas de renda alta têm **pouquíssimas observações**. Por exemplo, dizer que `6,19%` é diferente de `6,25%` não teria muito sentido aqui — são apenas **7 inadimplentes em cada grupo**.

**O que isso diz sobre o corte de R$ 50.000?**

Juntando tudo que investigamos:
```text
    Mediana                 R$     5.400
    P95                     R$    13.500
    P99                     R$    23.000
    P99,5                   R$    30.833
    Corte proposto          R$    50.000
    P99,9                   R$    72.838
    Máximo                  R$ 3.008.750
```
O corte de R$ 50 mil afeta apenas **225 dos 112.500 clientes de treino, ou 0,20%**. Além disso, não observamos nessa cauda um comportamento de inadimplência radicalmente diferente que sugira que precisamos preservar a magnitude de rendas de centenas de milhares ou milhões.

Por isso, **eu manteria R$ 50.000 como nosso teto operacional.**

Mas agora temos algo muito melhor do que simplesmente dizer:

> "Escolhemos R$ 50 mil porque parecia bom."

Temos uma justificativa:

> O teto de R$ 50.000 foi escolhido após análise exclusiva da base de treino. Ele preserva integralmente 99,8% das observações e reduz a influência de uma cauda extremamente longa, cujo máximo ultrapassa R$ 3 milhões. Critérios baseados em IQR ou percentis mais baixos produziriam cortes mais agressivos sobre rendas plausíveis.

Isso é uma justificativa metodológica bem mais sólida para documentarmos no projeto.

In [35]:
# 35. Aplicar o teto de R$ 50.000 à renda

X_treino["renda_mensal"] = X_treino["renda_mensal"].clip(upper=50000)
X_teste["renda_mensal"] = X_teste["renda_mensal"].clip(upper=50000)

In [36]:
# 36. Conferir o teto da renda após o tratamento

print("Máximo treino:", X_treino["renda_mensal"].max())
print("Máximo teste:", X_teste["renda_mensal"].max())

Máximo treino: 50000.0
Máximo teste: 50000.0


In [37]:
# 37. Criar a feature renda por dependente

X_treino["renda_por_dependente"] = (
    X_treino["renda_mensal"] / (X_treino["dependentes"] + 1)
)

X_teste["renda_por_dependente"] = (
    X_teste["renda_mensal"] / (X_teste["dependentes"] + 1)
)

**Considerando a renda por dependente**

Hoje temos separadamente:
```text
    renda_mensal
    dependentes
```
Imagine dois clientes:

| Cliente |    Renda | Dependentes |
| ------- | -------: | ----------: |
| A       | R$ 6.000 |           0 |
| B       | R$ 6.000 |           3 |


 

O modelo recebe a mesma renda para ambos, mas a situação financeira pode ser diferente. A ideia da nova feature é representar aproximadamente a **renda disponível por pessoa considerada nessa estrutura**:

```text
    Cliente A: 6.000 / (0 + 1) = R$ 6.000
    Cliente B: 6.000 / (3 + 1) = R$ 1.500
```
O `+ 1` representa o próprio cliente e também evita divisão por zero.

In [38]:
# 38. Conferir a nova feature renda por dependente

X_treino[
    ["renda_mensal", "dependentes", "renda_por_dependente"]
].head(10)

,renda_mensal,dependentes,renda_por_dependente
140300,8311.0,0.0,8311.000000
92417,5400.0,0.0,5400.000000
48193,3000.0,0.0,3000.000000
66644,1167.0,0.0,1167.000000
144734,5400.0,0.0,5400.000000
16152,5400.0,0.0,5400.000000
68997,7700.0,0.0,7700.000000
35575,14406.0,2.0,4802.000000
10305,4109.0,2.0,1369.666667
98645,3966.0,1.0,1983.000000


In [39]:
# 39. Criar a feature sobra de caixa

X_treino["sobra_caixa"] = (
    X_treino["renda_mensal"] * (1 - X_treino["razao_divida"])
)

X_teste["sobra_caixa"] = (
    X_teste["renda_mensal"] * (1 - X_teste["razao_divida"])
)

**Sobra de caixa**

A segunda feature proposta é:
```text
    sobra_caixa = renda_mensal × (1 - razao_divida)
```
Por exemplo, suponha:
```text
    renda_mensal = R$ 10.000
    razao_divida = 0,30
```
Teríamos:
```text
    10.000 × (1 - 0,30)
    = 10.000 × 0,70
    = R$ 7.000
```
A nova variável combina duas informações que estavam separadas:
```text
    renda + comprometimento com dívida
                    ↓
            sobra_caixa
```
Mas aqui há uma peculiaridade da nossa base: já vimos que `razao_divida` pode chegar a **2**. Portanto:
```text
    razao_divida = 1,2
    → sobra_caixa negativa

    razao_divida = 2
    → sobra_caixa = -renda
```
Isso **não é necessariamente um problema**. Uma sobra negativa pode carregar informação útil sobre uma situação de alto comprometimento. Não devemos truncá-la em zero automaticamente.

In [40]:
# 40. Conferir a nova feature sobra de caixa

X_treino[
    ["renda_mensal", "razao_divida", "sobra_caixa"]
].head(10)

,renda_mensal,razao_divida,sobra_caixa
140300,8311.0,0.004933,8270.004929
92417,5400.0,2.000000,-5400.000000
48193,3000.0,0.020327,2939.020326
66644,1167.0,0.642979,416.642980
144734,5400.0,2.000000,-5400.000000
16152,5400.0,2.000000,-5400.000000
68997,7700.0,0.180106,6313.180104
35575,14406.0,0.264385,10597.264389
10305,4109.0,0.353041,2658.353039
98645,3966.0,0.186539,3226.186540


In [41]:
# 41. Conferir as dimensões após os tratamentos

print("Treino:", X_treino.shape)
print("Teste:", X_teste.shape)

Treino: (112500, 15)
Teste: (37500, 15)


In [42]:
# 42. Conferir as colunas finais

X_treino.columns.tolist()

['idade',
 'renda_mensal',
 'dependentes',
 'uso_limite_rotativo',
 'razao_divida',
 'linhas_credito_abertas',
 'financiamentos_imobiliarios',
 'atrasos_30_59_dias',
 'atrasos_60_89_dias',
 'atrasos_90_mais_dias',
 'renda_estimada',
 'dependentes_estimados',
 'atrasos_96_98',
 'renda_por_dependente',
 'sobra_caixa']

In [43]:
# 43. Verificar valores ausentes após os tratamentos

X_treino.isna().sum()

idade                          0
renda_mensal                   0
dependentes                    0
uso_limite_rotativo            0
razao_divida                   0
linhas_credito_abertas         0
financiamentos_imobiliarios    0
atrasos_30_59_dias             0
atrasos_60_89_dias             0
atrasos_90_mais_dias           0
renda_estimada                 0
dependentes_estimados          0
atrasos_96_98                  0
renda_por_dependente           0
sobra_caixa                    0
dtype: int64

In [44]:
# 45. Conferir os tipos das variáveis

X_treino.dtypes

idade                            int64
renda_mensal                   float64
dependentes                    float64
uso_limite_rotativo            float64
razao_divida                   float64
linhas_credito_abertas           int64
financiamentos_imobiliarios      int64
atrasos_30_59_dias               int64
atrasos_60_89_dias               int64
atrasos_90_mais_dias             int64
renda_estimada                   int64
dependentes_estimados            int64
atrasos_96_98                    int64
renda_por_dependente           float64
sobra_caixa                    float64
dtype: object

In [45]:
# 46. Resumo estatístico após os tratamentos

X_treino.describe().T

,count,mean,std,min,25%,50%,75%,max
idade,112500.0,52.270373,14.762257,18.0,41.000000,52.000000,63.000000,103.000
renda_mensal,112500.0,6256.212089,4573.679983,0.0,3900.000000,5400.000000,7400.000000,50000.000
dependentes,112500.0,0.737831,1.106766,0.0,0.000000,0.000000,1.000000,20.000
uso_limite_rotativo,112500.0,0.324171,0.364828,0.0,0.029678,0.153517,0.558305,2.000
razao_divida,112500.0,0.680033,0.721403,0.0,0.175078,0.366299,0.857619,2.000
linhas_credito_abertas,112500.0,8.462951,5.150260,0.0,5.000000,8.000000,11.000000,58.000
financiamentos_imobiliarios,112500.0,1.018231,1.133211,0.0,0.000000,1.000000,2.000000,54.000
atrasos_30_59_dias,112500.0,0.281316,1.088853,0.0,0.000000,0.000000,0.000000,20.000
atrasos_60_89_dias,112500.0,0.100640,0.905565,0.0,0.000000,0.000000,0.000000,20.000
atrasos_90_mais_dias,112500.0,0.126729,0.972809,0.0,0.000000,0.000000,0.000000,20.000


In [46]:
# 47. Testes de consistência da base preparada

print("Valores ausentes:", X_treino.isna().sum().sum())

print(
    "Rendas acima de 50.000:",
    (X_treino["renda_mensal"] > 50000).sum()
)

print(
    "Códigos 96/98 restantes:",
    X_treino[colunas_atraso].isin([96, 98]).sum().sum()
)

print(
    "Maior renda:",
    X_treino["renda_mensal"].max()
)

print(
    "Maiores valores de atraso:"
)
print(X_treino[colunas_atraso].max())

Valores ausentes: 0
Rendas acima de 50.000: 0
Códigos 96/98 restantes: 0
Maior renda: 50000.0
Maiores valores de atraso:
atrasos_30_59_dias      20
atrasos_60_89_dias      20
atrasos_90_mais_dias    20
dtype: int64


In [47]:
# 48. Validar o cálculo da feature renda_por_dependente

renda_por_dependente_calculada = (
    X_treino["renda_mensal"] / (X_treino["dependentes"] + 1)
)

erros_renda_por_dependente = (
    X_treino["renda_por_dependente"] != renda_por_dependente_calculada
).sum()

print("Inconsistências:", erros_renda_por_dependente)

Inconsistências: 0


In [48]:
# 49. Validar o cálculo da feature sobra_caixa

sobra_caixa_calculada = (
    X_treino["renda_mensal"] * (1 - X_treino["razao_divida"])
)

erros_sobra_caixa = (
    X_treino["sobra_caixa"] != sobra_caixa_calculada
).sum()

print("Inconsistências:", erros_sobra_caixa)

Inconsistências: 0


In [49]:
# 50. Preservar o resultado do tratamento manual

X_treino_manual = X_treino.copy()
X_teste_manual = X_teste.copy()

In [50]:
# 51. Importar a nova função

from consumer_credit_risk.preprocessing import preparar_dados_credito


In [51]:
# 52. Recarregar os dados originais

df_original = carregar_dados_credito()

X_original = df_original.drop(columns="inadimplente_2anos")
y_original = df_original["inadimplente_2anos"]

In [52]:
# mesma divisao usada anteriormente

X_treino_original, X_teste_original, y_treino_original, y_teste_original = (
    train_test_split(
        X_original,
        y_original,
        test_size=0.25,
        random_state=42,
        stratify=y_original,
    )
)

O `random_state=42` é fundamental aqui. Como repetimos exatamente a mesma divisão, os mesmos clientes devem cair no treino e no teste.

In [53]:
# 53. Preparar os dados utilizando preprocessing.py

X_treino_preparado = preparar_dados_credito(
    X_treino_original
)

X_teste_preparado = preparar_dados_credito(
    X_teste_original
)

In [54]:
# 54. Verificar as dimensões após o preparo

print("Treino antes do preparo:", X_treino_original.shape)
print("Treino após o preparo:", X_treino_preparado.shape)

print()

print("Teste antes do preparo:", X_teste_original.shape)
print("Teste após o preparo:", X_teste_preparado.shape)

Treino antes do preparo: (112500, 10)
Treino após o preparo: (112500, 15)

Teste antes do preparo: (37500, 10)
Teste após o preparo: (37500, 15)


In [55]:
# 55. Verificar valores ausentes após o preparo

print("Valores ausentes no treino:")
print(X_treino_preparado.isna().sum())

print()

print("Valores ausentes no teste:")
print(X_teste_preparado.isna().sum())

Valores ausentes no treino:
idade                              0
renda_mensal                   22187
dependentes                     2945
uso_limite_rotativo                0
razao_divida                       0
linhas_credito_abertas             0
financiamentos_imobiliarios        0
atrasos_30_59_dias                 0
atrasos_60_89_dias                 0
atrasos_90_mais_dias               0
renda_estimada                     0
dependentes_estimados              0
atrasos_96_98                      0
renda_por_dependente           22187
sobra_caixa                    22187
dtype: int64

Valores ausentes no teste:
idade                             0
renda_mensal                   7544
dependentes                     979
uso_limite_rotativo               0
razao_divida                      0
linhas_credito_abertas            0
financiamentos_imobiliarios       0
atrasos_30_59_dias                0
atrasos_60_89_dias                0
atrasos_90_mais_dias              0
renda_estimada  

In [56]:
# 56. Importar o SimpleImputer

from sklearn.impute import SimpleImputer

In [57]:
# 57. Definir as colunas para imputação

colunas_imputacao = [
    "renda_mensal",
    "dependentes",
    "renda_por_dependente",
    "sobra_caixa",
]

In [58]:
# 58. Criar e ajustar o SimpleImputer

imputer = SimpleImputer(strategy="median")

imputer.fit(
    X_treino_preparado[colunas_imputacao]
)

,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](4,)","['renda_mensal','dependentes','renda_por_dependente','sobra_caixa']"
indicator_ indicator_: :class:`~sklearn.impute.MissingIndicator`Indicator used to add binary indicators for missing values.`None` if `add_indicator=False`.,NoneType,None
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,4
"statistics_ statistics_: array of shape (n_features,)The imputation fill value for each feature.Computing statistics can result in `np.nan` values.During :meth:`transform`, features corresponding to `np.nan`statistics will be discarded.","ndarray[float64](4,)","[5400. , 0. ,3333.33,3571.24]"


In [59]:
dict(zip(colunas_imputacao, imputer.statistics_))

{'renda_mensal': np.float64(5400.0),
 'dependentes': np.float64(0.0),
 'renda_por_dependente': np.float64(3333.3333333333335),
 'sobra_caixa': np.float64(3571.242739236)}

In [60]:
# 59. Aplicar a imputação ao treino e ao teste

X_treino_imputado = X_treino_preparado.copy()
X_teste_imputado = X_teste_preparado.copy()

X_treino_imputado[colunas_imputacao] = imputer.transform(
    X_treino_preparado[colunas_imputacao]
)

X_teste_imputado[colunas_imputacao] = imputer.transform(
    X_teste_preparado[colunas_imputacao]
)

In [61]:
# 60. Verificar valores ausentes após a imputação

print(
    "Valores ausentes no treino:",
    X_treino_imputado.isna().sum().sum()
)

print(
    "Valores ausentes no teste:",
    X_teste_imputado.isna().sum().sum()
)

Valores ausentes no treino: 0
Valores ausentes no teste: 0


In [62]:
# 61. Importar as ferramentas para construção do pipeline

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

In [63]:
# 62. Criar o transformador de preparo

preparo = FunctionTransformer(
    preparar_dados_credito,
    validate=False
)

In [64]:
# 63. Testar o transformador de preparo

X_treino_transformado = preparo.fit_transform(
    X_treino_original
)

print("Antes:", X_treino_original.shape)
print("Depois:", X_treino_transformado.shape)

Antes: (112500, 10)
Depois: (112500, 15)


In [65]:
pd.testing.assert_frame_equal(
    X_treino_preparado,
    X_treino_transformado
)

print("FunctionTransformer reproduziu corretamente a função de preparo.")

FunctionTransformer reproduziu corretamente a função de preparo.


In [66]:
# 64. Importar o ColumnTransformer

from sklearn.compose import ColumnTransformer

In [68]:
# 65. Criar a etapa de imputação

imputacao = ColumnTransformer(
    transformers=[
        (
            "imputer_mediana",
            SimpleImputer(strategy="median"),
            colunas_imputacao,
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

In [69]:
# 66. Criar o pipeline de pré-processamento

pipeline_preprocessamento = Pipeline(
    steps=[
        ("preparo", preparo),
        ("imputacao", imputacao),
    ]
)

Neste momento temos:
```text
X original
    │
    ▼
┌──────────────────────────────┐
│ 1. preparo                   │
│ preparar_dados_credito()     │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│ 2. imputacao                 │
│ SimpleImputer(median)        │
└──────────────┬───────────────┘
               │
               ▼
        dados para o modelo
```
Ainda não colocamos a terceira etapa:
```text
    3. modelo
```
porque ainda vamos definir quais modelos serão usados. Depois poderemos acrescentá-la sem modificar as duas etapas que estamos construindo agora.

In [70]:
# 67. Aplicar o pipeline de pré-processamento

X_treino_pipeline = pipeline_preprocessamento.fit_transform(
    X_treino_original
)

X_teste_pipeline = pipeline_preprocessamento.transform(
    X_teste_original
)

In [71]:
# 68. Validar o resultado do pipeline

print("Dimensão do treino:", X_treino_pipeline.shape)
print("Dimensão do teste:", X_teste_pipeline.shape)

print()

print(
    "Valores ausentes no treino:",
    X_treino_pipeline.isna().sum().sum()
)

print(
    "Valores ausentes no teste:",
    X_teste_pipeline.isna().sum().sum()
)

Dimensão do treino: (112500, 15)
Dimensão do teste: (37500, 15)

Valores ausentes no treino: 0
Valores ausentes no teste: 0


Se isso acontecer, teremos comprovado que o pipeline recebe os 10 atributos originais e automaticamente:
```text
    10 features originais
            ↓
        PREPARO
            ↓
    15 features
    (com alguns NaN)
            ↓
        IMPUTAÇÃO
            ↓
    15 features
    (sem NaN)
```
E repare em algo importante: agora não precisamos mais fazer manualmente.

In [72]:
# 69. Verificar as features após o pipeline

print("Número de features:", len(X_treino_pipeline.columns))

print()

for coluna in X_treino_pipeline.columns:
    print(coluna)

Número de features: 15

renda_mensal
dependentes
renda_por_dependente
sobra_caixa
idade
uso_limite_rotativo
razao_divida
linhas_credito_abertas
financiamentos_imobiliarios
atrasos_30_59_dias
atrasos_60_89_dias
atrasos_90_mais_dias
renda_estimada
dependentes_estimados
atrasos_96_98


In [73]:
# 70. Verificar as medianas aprendidas pelo pipeline

imputer_pipeline = (
    pipeline_preprocessamento
    .named_steps["imputacao"]
    .named_transformers_["imputer_mediana"]
)

medianas_pipeline = dict(
    zip(
        colunas_imputacao,
        imputer_pipeline.statistics_
    )
)

medianas_pipeline

{'renda_mensal': np.float64(5400.0),
 'dependentes': np.float64(0.0),
 'renda_por_dependente': np.float64(3333.3333333333335),
 'sobra_caixa': np.float64(3571.242739236)}

In [74]:
# 71. Comparar o resultado manual com o resultado do pipeline

X_treino_pipeline_ordenado = X_treino_pipeline[
    X_treino_imputado.columns
]

pd.testing.assert_frame_equal(
    X_treino_imputado,
    X_treino_pipeline_ordenado,
    check_dtype=False
)

print("Pipeline reproduziu corretamente o processo manual.")

Pipeline reproduziu corretamente o processo manual.


In [75]:
# 72. Criar função para montar o pipeline completo

def criar_pipeline(modelo):
    return Pipeline(
        steps=[
            ("preparo", preparo),
            ("imputacao", imputacao),
            ("modelo", modelo),
        ]
    )